# **Cross-Dataset Experiment (ACDC → DAWN) — YOLOv11s**
- Training YOLOv11s on the ACDC dataset using COCO pretrained weights
- Evaluating the trained model on the global DAWN test set
- Performing weather-specific evaluations on the fog, rain, and snow subsets of the DAWN dataset

## Mount drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install Ultralytics

In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.4 MB/s eta 0:00:00


## Import librairies

In [4]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import time
import torch

# Check GPU

In [5]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-40GB


## Path

In [6]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

RUNS_ROOT = PROJECT_ROOT / "Runs"
DATASETS_ROOT = PROJECT_ROOT / "Datasets" / "processed"

# ACDC (training dataset)
ACDC_YOLO_ROOT = DATASETS_ROOT / "acdc_yolo"

# DAWN (evaluation) dataset)
DAWN_YOLO_ROOT = DATASETS_ROOT / "dawn_yolo"


## YAML Files

In [7]:
# ACDC training YAML
ACDC_DATA_YAML = ACDC_YOLO_ROOT / "acdc.yaml"

# DAWN  evaluation YAMLs
DAWN_GLOBAL_YAML = DAWN_YOLO_ROOT / "dataset.yaml"
DAWN_FOG_YAML    = DAWN_YOLO_ROOT / "fog_only.yaml"
DAWN_RAIN_YAML   = DAWN_YOLO_ROOT / "rain_only.yaml"
DAWN_SNOW_YAML   = DAWN_YOLO_ROOT / "snow_only.yaml"

## Classes

In [8]:
CLASS_NAMES = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "truck"
]

## Experiment Settings

In [9]:
SEED = 456

MODEL_NAME = "yolo11s.pt"

EXPERIMENT_NAME = f"acdc_to_dawn_yolo11s_seed{SEED}"

## Training

In [10]:
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(ACDC_DATA_YAML),

    # Training
    epochs=100,
    patience=10,

    # Image / batch
    imgsz=640,
    batch=16,

    # Reproducibility
    seed=SEED,
    deterministic=True,
    pretrained=True,

    # Validation
    val=True,

    # Workers
    workers=2,

    # Output
    project=str(RUNS_ROOT / "yolo"),
    name=EXPERIMENT_NAME,
    exist_ok=True
)

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Dissertation/Datasets/processed/acdc_yolo/acdc.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=acdc_to_dawn_yolo11s_seed45

## Load best model

In [11]:
BEST_MODEL_PATH = (
    RUNS_ROOT
    / "yolo"
    / EXPERIMENT_NAME
    / "weights"
    / "best.pt"
)

best_model = YOLO(str(BEST_MODEL_PATH))

print(f"\nBest model loaded from:\n{BEST_MODEL_PATH}")


Best model loaded from:
/content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed456/weights/best.pt


## Evaluation Function

In [12]:
OUTPUT_DIR = RUNS_ROOT / "yolo" / EXPERIMENT_NAME / "cross_dataset_eval_dawn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Evaluation outputs will be saved to:\n{OUTPUT_DIR}")

Evaluation outputs will be saved to:
/content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed456/cross_dataset_eval_dawn


In [13]:
def evaluate_split(model, yaml_path, split_name):

    print(f"\nEvaluating ACDC -> DAWN on: {split_name}")

    metrics = model.val(
        data=str(yaml_path),
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        verbose=False,
        project=str(OUTPUT_DIR),
        name=f"eval_{split_name}",
        exist_ok=True
    )

    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)

    f1_score = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    preprocess_ms = float(metrics.speed.get("preprocess", 0.0))
    inference_ms = float(metrics.speed.get("inference", 0.0))
    postprocess_ms = float(metrics.speed.get("postprocess", 0.0))

    total_ms_per_image = preprocess_ms + inference_ms + postprocess_ms
    fps = 1000 / total_ms_per_image if total_ms_per_image > 0 else 0.0

    return {
        "source_train_dataset": "ACDC",
        "target_test_dataset": "DAWN",
        "model": "YOLOv11s",
        "experiment": "ACDC->DAWN",
        "seed": SEED,
        "condition": split_name,

        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "mAP75": float(metrics.box.map75),

        "Precision": precision,
        "Recall": recall,
        "F1-score": f1_score,

        "Preprocess_ms_per_image": preprocess_ms,
        "Inference_ms_per_image": inference_ms,
        "Postprocess_ms_per_image": postprocess_ms,
        "Total_ms_per_image": total_ms_per_image,
        "FPS": fps,
    }

## Global and weather-specific evaluation

In [16]:
import yaml

weather_yaml_updates = {
    DAWN_FOG_YAML: "images/test_fog",
    DAWN_RAIN_YAML: "images/test_rain",
    DAWN_SNOW_YAML: "images/test_snow",
}

for yaml_path, test_path in weather_yaml_updates.items():
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    data["test"] = test_path

    with open(yaml_path, "w") as f:
        yaml.dump(data, f, sort_keys=False)

    print(f"Updated {yaml_path}: test = {test_path}")

Updated /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/fog_only.yaml: test = images/test_fog
Updated /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/rain_only.yaml: test = images/test_rain
Updated /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/snow_only.yaml: test = images/test_snow


In [17]:
evaluation_configs = [
    ("global", DAWN_GLOBAL_YAML),
    ("fog", DAWN_FOG_YAML),
    ("rain", DAWN_RAIN_YAML),
    ("snow", DAWN_SNOW_YAML),
]

all_results = []

for split_name, yaml_path in evaluation_configs:

    print(f"\nEvaluating ACDC -> DAWN on: {split_name}")

    result = evaluate_split(
        model=best_model,
        yaml_path=yaml_path,
        split_name=split_name
    )

    all_results.append(result)


Evaluating ACDC -> DAWN on: global

Evaluating ACDC -> DAWN on: global
Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
val: Fast image access ✅ (ping: 6.1±12.3 ms, read: 53.1±44.2 MB/s, size: 242.7 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test.cache... 140 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 140/140 45.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.9it/s 1.8s
                   all        140        920      0.535      0.144       0.17     0.0957
Speed: 1.2ms preprocess, 1.2ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed456/cross_dataset_eval_dawn/eval_global

Evaluating ACDC -> DAWN on: fog

Evaluating A

In [18]:
for name, yaml_path in evaluation_configs:
    print("\n", name, yaml_path)
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    print(data)



 global /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/dataset.yaml
{'path': '/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo', 'train': 'images/train', 'val': 'images/val', 'nc': 6, 'names': ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck'], 'test': 'images/test'}

 fog /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/fog_only.yaml
{'path': '/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo', 'train': 'images/train', 'val': 'images/test_fog', 'nc': 6, 'names': ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck'], 'test': 'images/test_fog'}

 rain /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/rain_only.yaml
{'path': '/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo', 'train': 'images/train', 'val': 'images/test_rain', 'nc': 6, 'names': ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck'], 'test': 'images/test_rain'}

 snow /content/drive/MyDrive/Dissertation/Datase

## Results dataframe

In [19]:
results_df = pd.DataFrame(all_results)

print("ACDC -> DAWN YOLOv11s RESULTS")

display(results_df)

ACDC -> DAWN YOLOv11s RESULTS


,source_train_dataset,target_test_dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Preprocess_ms_per_image,Inference_ms_per_image,Postprocess_ms_per_image,Total_ms_per_image,FPS
0,ACDC,DAWN,YOLOv11s,ACDC->DAWN,456,global,0.095681,0.170482,0.082078,0.534970,0.143990,0.226906,1.227754,1.248263,2.048514,4.524531,221.017386
1,ACDC,DAWN,YOLOv11s,ACDC->DAWN,456,fog,0.084670,0.151802,0.071376,0.582977,0.126378,0.207726,2.267808,4.042696,2.488924,8.799428,113.643748
2,ACDC,DAWN,YOLOv11s,ACDC->DAWN,456,rain,0.112762,0.223759,0.105229,0.897904,0.199107,0.325939,2.501054,4.934740,5.652967,13.088762,76.401422
3,ACDC,DAWN,YOLOv11s,ACDC->DAWN,456,snow,0.169911,0.279182,0.155672,0.397751,0.221434,0.284489,2.503272,1.543840,5.125765,9.172878,109.017044


## Save results

In [ ]:
RESULTS_DIR = (
    RUNS_ROOT
    / "yolo"
    / EXPERIMENT_NAME
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

csv_path = RESULTS_DIR / "acdc_to_dawn_yolo11s_results.csv"
json_path = RESULTS_DIR / "acdc_to_dawn_yolo11s_results.json"
xlsx_path = RESULTS_DIR / "acdc_to_dawn_yolo11s_results.xlsx"

results_df.to_csv(csv_path, index=False)

results_df.to_json(
    json_path,
    orient="records",
    indent=4
)

results_df.to_excel(
    xlsx_path,
    index=False
)

print("\nResults saved:")
print(f"CSV  : {csv_path}")
print(f"JSON : {json_path}")
print(f"XLSX : {xlsx_path}")


Results saved:
CSV  : /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed42/acdc_to_dawn_yolo11s_results.csv
JSON : /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed42/acdc_to_dawn_yolo11s_results.json
XLSX : /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed42/acdc_to_dawn_yolo11s_results.xlsx


## Save summary

In [ ]:
summary_path = RESULTS_DIR / "experiment_summary.txt"

with open(summary_path, "w") as f:
    f.write("YOLOv11s Cross-Dataset Experiment\n")
    f.write(f"Experiment: {EXPERIMENT_NAME}\n")
    f.write("Direction: ACDC -> DAWN\n")
    f.write("Source training dataset: ACDC\n")
    f.write("Target evaluation dataset: DAWN\n")
    f.write("Model: YOLOv11s\n")
    f.write(f"Seed: {SEED}\n")
    f.write("Training epochs: 100\n")
    f.write("Early stopping patience: 10\n")
    f.write("Image size: 640\n")
    f.write("Batch size: 16\n")
    f.write("Evaluation: global + fog + rain + snow\n")
    f.write(f"\nBest model path:\n{BEST_MODEL_PATH}\n")
    f.write(f"\nResults CSV:\n{csv_path}\n")
    f.write(f"\nResults JSON:\n{json_path}\n")
    f.write(f"\nResults XLSX:\n{xlsx_path}\n")

print(f"Summary saved: {summary_path}")

Summary saved: /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_to_dawn_yolo11s_seed42/experiment_summary.txt
